# Qwen2.5-VL on Colab

Runs Qwen2.5-VL with the authors' prompt on (1) the Polish dataset and (2) the 330-image Turkish replication sample. Same code as the local runs (`src/run_ocr.py`), so the output CSVs drop straight into `polish/results/` and `replication/results/`.

1. Runtime -> Change runtime type -> GPU. **L4 or A100** loads the 7B model used by the authors; a **T4** falls back to 3B (record which one in the article).
2. Locally run `colab/make_bundle.sh`, then upload `colab/ocr_bundle.zip` when the cell below asks.
3. Runs are resumable: if Colab disconnects, re-upload the CSVs from Drive (or just rerun; finished images are skipped).

In [ ]:
!pip -q install qwen-vl-utils editdistance jiwer
import os
from google.colab import files
if not os.path.exists('ocr_bundle.zip'):
    files.upload()
!unzip -qo ocr_bundle.zip && ls

In [ ]:
# Optional: keep results on Google Drive so a disconnect loses nothing
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ocr-benchmark'
os.makedirs(OUT, exist_ok=True)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# None = automatic: 7B if the GPU has > 20 GB, else 3B. Force with e.g. 'Qwen/Qwen2.5-VL-7B-Instruct'.
CHECKPOINT = None
ckpt = f'--checkpoint {CHECKPOINT}' if CHECKPOINT else ''

In [ ]:
# 1. Polish dataset (440 images)
!python src/run_ocr.py --model qwen --lang Polish {ckpt} --data polish/data --out {OUT}/qwen_polish.csv

In [ ]:
# 2. Turkish replication sample (330 images), authors' original prompt
!python src/run_ocr.py --model qwen --lang Turkish {ckpt} --data replication/data --labels sample.csv --out {OUT}/qwen_turkish.csv

In [ ]:
# Download and rename: qwen_polish.csv -> polish/results/qwen7b.csv, qwen_turkish.csv -> replication/results/qwen7b.csv
# (use qwen.csv instead of qwen7b.csv if the GPU was too small and the 3B model ran)
files.download(f'{OUT}/qwen_polish.csv')
files.download(f'{OUT}/qwen_turkish.csv')